# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring — not classification.**

I originally framed this as binary classification (`is_page_1`, using the small starter
sample). That was a mismatch I only caught once ML-04 and ML-07 were built: ML-02 already
defined the output of this lane as *"a ranked list of pages ordered by refresh priority"* —
that's a ranking task, not a two-bucket classification. Two more concrete reasons ranking is
the better fit, once you look at the real warehouse data:

1. **A cutoff throws away information a content team actually needs.** Two pages could both
   sit outside "page 1," but one is at position 12 and the other at position 45 — very
   different opportunities. A ranking score preserves that difference; `is_page_1` collapses it
   to the same "0."
2. **The baseline (ML-07) already works this way and it's real, running code** — it produces a
   continuous `action_score` per page and sorts by it. Framing ML-03 as classification would
   describe a different project than the one actually being built from ML-04 onward.


In [1]:
# Section 1 has no computation of its own — the task-type decision is argued above,
# and backed by what ML-02 already committed to (a ranked list) and what ML-07 already built.
print("Task type: ranking / scoring (score = ctr_gap, business-weighted by impressions)")


Task type: ranking / scoring (score = ctr_gap, business-weighted by impressions)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target: `ctr_gap` (continuous), same construct ML-04 defined and ML-07 already computes.**

`ctr_gap = expected_ctr_for_this_page's_position_bucket − actual_ctr`, at content-page grain,
for the current month. This is an **observed** quantity — built directly from real GSC
impressions, clicks, and position — not an invented proxy. `action_score = ctr_gap ×
impressions` (ML-07's business-weighted version) is what actually gets ranked, since a small
gap on a huge-traffic page usually matters more than a big gap on a tiny one.

**What's still an open decision, honestly:** the baseline computes `ctr_gap` directly from this
month's own numbers — a model can't "predict" something already computed from data it would be
handed as input. What ML-08 actually needs to do is either (a) predict `ctr_gap` from features
that exclude this month's GSC numbers (so it's useful before/without a full month of fresh
data), or (b) predict *future* decline using `prev30 → last30` transitions from ML-04's table.
I'm not resolving that here — noting it so it isn't quietly decided by default when we scope
ML-08. This section commits to the target **concept** (CTR gap vs. position), not the final
training recipe.

One honest caveat carried over from the caveat in the original version of this section: this
is a snapshot of *current* underperformance, not a guarantee that fixing it moves the numbers.
Decision-support, not a promise.


In [3]:
import pandas as pd
import numpy as np

# Parquet files are in the project root
DATA_DIR = "../.."

perf = pd.read_parquet(
    f"{DATA_DIR}/fact_content_daily_performance_sample.parquet",
    columns=[
        "report_date",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_avg_position"
    ]
)

dc = pd.read_parquet(
    f"{DATA_DIR}/dim_content.parquet",
    columns=[
        "client_hash_id",
        "content_hash_id",
        "search_volume",
        "is_published",
        "is_deleted"
    ]
)

agg = perf.groupby(
    ["client_hash_id", "content_hash_id"]
).agg(
    impressions=("gsc_impressions", "sum"),
    clicks=("gsc_clicks", "sum"),
    avg_pos=("gsc_avg_position", "mean"),
).reset_index()

agg = agg[agg.impressions >= 50].copy()

agg["ctr"] = agg.clicks / agg.impressions

bins = [0, 3, 5, 10, 20, 50, 1000]
labels = ["1-3", "3-5", "5-10", "10-20", "20-50", "50+"]

agg["pos_bucket"] = pd.cut(
    agg.avg_pos,
    bins=bins,
    labels=labels
).astype(str)

expected_ctr_by_bucket = agg.groupby(
    "pos_bucket",
    observed=True
).apply(
    lambda g: g.clicks.sum() / g.impressions.sum()
)

agg["expected_ctr"] = agg.pos_bucket.map(
    expected_ctr_by_bucket
).astype(float)

agg["ctr_gap"] = agg.expected_ctr - agg.ctr

print("Rows (content pages, impressions>=50):", len(agg))

print("\nctr_gap distribution:")
print(agg.ctr_gap.describe())

Rows (content pages, impressions>=50): 120681

ctr_gap distribution:
count    120679.000000
mean          0.000055
std           0.008344
min          -0.492624
25%          -0.001964
50%           0.000722
75%           0.002632
max           0.044121
Name: ctr_gap, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Success metric: Precision@K (primary), e.g. Precision@50 — not ROC-AUC/F1.**

ROC-AUC and F1 are built for a two-class target; `ctr_gap` is continuous and the task is
ranking, so they don't fit anymore. Precision@K matches both the repo's own convention (the
reference pipeline's committed baseline is reported as **Precision@50 = 0.240**, per
`GUIDE.md`) and the real business use: a content team only has time to review a fixed number
of pages, so what matters is "of the top K pages the score picks, how many are actually good
picks" — not how well the model separates all pages across every possible threshold.

**Open question, not resolved here:** Precision@K needs a real yes/no "good pick" outcome to
check against, and right now `ctr_gap` and "good pick" are the same number — there's no
independent ground truth yet (e.g. content that was actually reviewed and improved). ML-08/09
will need to define what counts as a correct pick — most likely, rank agreement with the
baseline as a sanity check, and/or observed decline reversal on a later time slice if one
becomes available. Flagging this now so it's a deliberate decision later, not a gap that gets
missed.


In [4]:
# There's no majority-class baseline for a continuous target. What IS checkable here:
# does re-running the aggregation reproduce ML-07's committed queue? (a basic sanity check,
# not a metric) — compare row count and top-1 action_score to work/outputs/baseline_action_score.csv
import os
existing = "work/outputs/baseline_action_score.csv"
if os.path.exists(existing):
    prev = pd.read_csv(existing)
    print("ML-07 queue rows:", len(prev), "| top action_score:", prev.action_score.max())
else:
    print(f"{existing} not found yet — run ML-07 first, then re-run this cell to sanity-check consistency.")


work/outputs/baseline_action_score.csv not found yet — run ML-07 first, then re-run this cell to sanity-check consistency.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of analysis: one row = one content page (`client_hash_id` + `content_hash_id`), this month.**

Matches ML-02's stated unit ("one content page") and ML-07's grain exactly — not the toy
CSV's `content_id` row, which was a different dataset entirely. `impressions`, `clicks`, and
`avg_pos` are summed/averaged across all of a page's queries and days within the month (from
`fact_content_daily_performance_sample`); `ctr_gap` is computed on top of that, as shown below.


In [5]:
print("Shape:", agg.shape)
agg[["client_hash_id","content_hash_id","avg_pos","pos_bucket","ctr","expected_ctr","ctr_gap"]].head(10)


Shape: (120681, 9)


,client_hash_id,content_hash_id,avg_pos,pos_bucket,ctr,expected_ctr,ctr_gap
1218,client_06d356715a8ff3b6,content_0058bd88fb1821f2,10.405118,10-20,0.004149,0.004102,-0.000047
1219,client_06d356715a8ff3b6,content_0059a4d4195810c9,12.305041,10-20,0.002367,0.004102,0.001735
1220,client_06d356715a8ff3b6,content_005b6b7f7b8dda7f,8.003647,5-10,0.003135,0.003530,0.000396
1221,client_06d356715a8ff3b6,content_0094c7d0fbcc07b7,26.829830,20-50,0.000000,0.002632,0.002632
1222,client_06d356715a8ff3b6,content_00a34394d4ee05ce,17.623590,10-20,0.007042,0.004102,-0.002940
1223,client_06d356715a8ff3b6,content_00fd1bae29d73213,8.335814,5-10,0.011236,0.003530,-0.007706
1224,client_06d356715a8ff3b6,content_012da01a3ee631b5,16.326142,10-20,0.005882,0.004102,-0.001780
1225,client_06d356715a8ff3b6,content_0153b7dedc3fc40d,16.397196,10-20,0.004317,0.004102,-0.000214
1226,client_06d356715a8ff3b6,content_0170639e18314d31,6.363758,5-10,0.009671,0.003530,-0.006141
1227,client_06d356715a8ff3b6,content_01ad5f3e74c28a0d,10.594463,10-20,0.002509,0.004102,0.001593


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**A fixed rule only gets to use one signal by design — real candidate signals it's ignoring are sitting unused in `dim_content`.**

ML-07's rule deliberately uses exactly one signal (`ctr_gap` vs. position) — that's the point
of a transparent baseline: one reason code, human-readable. But `dim_content` has other
page-level signals never touched by that rule: `search_volume`, `competition`, `cpc`,
`backlinks`, `word_count`, `main_intent`, `content_type`. A model could combine these with
`ctr_gap` to catch patterns a single if/then rule can't express — e.g. "large CTR gap + high
search volume + low competition" might deserve to outrank a raw `ctr_gap`-sorted page that's
already near-maximally competitive.

**I have NOT filled in a conclusion below — that's for you to write after running the cell and
looking at the real numbers.** Don't write "confirmed" or "this proves ML helps" until you've
actually seen a correlation that supports it. A flat or near-zero correlation across the board
is a legitimate outcome here too, same as Signal 2 was `FALSE` in ML-07 — if that's what the
numbers show, say so.


In [6]:
dc_full = pd.read_parquet(f"{DATA_DIR}/dim_content.parquet",
    columns=["client_hash_id","content_hash_id","search_volume","competition","cpc",
             "backlinks","word_count","is_published","is_deleted"])
dc_full = dc_full[dc_full.is_published & ~dc_full.is_deleted]

check = agg.merge(dc_full, on=["client_hash_id","content_hash_id"], how="inner")

candidate_cols = ["search_volume", "competition", "cpc", "backlinks", "word_count"]
print("Correlation of each candidate feature with ctr_gap:")
print(check[candidate_cols + ["ctr_gap"]].corr()["ctr_gap"].drop("ctr_gap").sort_values())
print()
print("Missing-value counts among candidates (matters for whether they're usable as-is):")
print(check[candidate_cols].isnull().sum())

# >>> RUN THIS CELL, then write your actual verdict in the markdown cell above based on
# >>> what prints here — do not write a conclusion before you have the real numbers.


Correlation of each candidate feature with ctr_gap:
word_count      -0.023089
backlinks        0.000646
cpc              0.001347
competition      0.002038
search_volume    0.005989
Name: ctr_gap, dtype: float64

Missing-value counts among candidates (matters for whether they're usable as-is):
search_volume     1630
competition       1630
cpc               1630
backlinks        33343
word_count       24511
dtype: int64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Section 5's conclusion reflects the real printed correlation numbers, not an assumption
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
